Step 1: Load Dataset
Upload and load the AI jobs dataset into a Pandas DataFrame for further preprocessing and analysis.

In [30]:

from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)
df.shape
df.head()

Saving ai_jobs_market_2025_2026.csv to ai_jobs_market_2025_2026 (1).csv


,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,...,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7,Master's,239000.0,155000,290000,Boston,...,13.1,96,16.9,6.8,2026,3,1,0,1,Senior ($200-300k)
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2,Bachelor's,166000.0,90000,200000,London,...,5.4,82,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k)
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4,Associate's,360000.0,160000,300000,Seattle,...,9.1,98,42.7,7.7,2026,1,1,1,1,Elite (>$300k)
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3,Bachelor's,161000.0,130000,220000,Singapore,...,12.0,88,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k)
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,...,9.4,85,17.3,8.9,2026,1,1,1,0,Senior ($200-300k)


Step 2: Inspect Dataset Structure
Examine the dataset dimensions, column names, and data types to understand its initial structure.

In [31]:

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Dataset Shape: (1500, 25)

Column Names:
['job_id', 'job_title', 'job_category', 'experience_level', 'years_of_experience', 'education_required', 'annual_salary_usd', 'salary_min_usd', 'salary_max_usd', 'city', 'country', 'remote_work', 'company_size', 'industry', 'required_skills', 'ai_salary_premium_pct', 'demand_score', 'demand_growth_yoy_pct', 'benefits_score_10', 'posting_year', 'posting_month', 'is_senior', 'is_remote_friendly', 'is_llm_role', 'salary_tier']

Data Types:
job_id                    object
job_title                 object
job_category              object
experience_level          object
years_of_experience        int64
education_required        object
annual_salary_usd        float64
salary_min_usd             int64
salary_max_usd             int64
city                      object
country                   object
remote_work               object
company_size              object
industry                  object
required_skills           object
ai_salary_premium_pct  

Step 3: Check Missing Values
Identify missing values in each column to determine whether additional missing-value treatment is required.

In [32]:

missing_values = df.isnull().sum()

missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Values': missing_values,
    'Missing Percentage': missing_percentage
})

missing_summary.sort_values(
    by='Missing Percentage',
    ascending=False
)

,Missing Values,Missing Percentage
job_id,0,0.0
job_title,0,0.0
job_category,0,0.0
experience_level,0,0.0
years_of_experience,0,0.0
education_required,0,0.0
annual_salary_usd,0,0.0
salary_min_usd,0,0.0
salary_max_usd,0,0.0
city,0,0.0


Step 4: Check Duplicate Records
Identify duplicated rows to prevent repeated records from affecting statistical analysis and visualizations.

In [33]:

duplicate_count = df.duplicated().sum()

print("Number of Duplicate Rows:", duplicate_count)

Number of Duplicate Rows: 0


Step 5: Clean Text Whitespace
Remove unnecessary leading and trailing whitespace from text columns to ensure consistent categorical values.

In [34]:

text_cols = df.select_dtypes(include='object').columns

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

Step 6: Standardize Missing-Like Values
Convert common textual representations of missing values into actual NaN values for consistent data handling

In [35]:

missing_like_values = [
    '',
    ' ',
    'N/A',
    'n/a',
    'NA',
    'na',
    'None',
    'none',
    'Unknown',
    'unknown',
    'Not Available',
    'not available'
]

df.replace(missing_like_values, np.nan, inplace=True)

Step 7: Convert Date Columns
Convert date-related columns into appropriate datetime format to enable time-based analysis and visualization.

In [36]:

date_cols = [
    col for col in df.columns
    if 'date' in col.lower()
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(df[date_cols].dtypes)

Series([], dtype: object)


Step 8: Validate Numeric Columns
Identify numeric columns and inspect their statistical ranges to detect invalid or unrealistic values before visualization.

In [37]:

numeric_cols = df.select_dtypes(include=np.number).columns

print("Numeric Columns:")
print(numeric_cols.tolist())

df[numeric_cols].describe().T

Numeric Columns:
['years_of_experience', 'annual_salary_usd', 'salary_min_usd', 'salary_max_usd', 'ai_salary_premium_pct', 'demand_score', 'demand_growth_yoy_pct', 'benefits_score_10', 'posting_year', 'posting_month', 'is_senior', 'is_remote_friendly', 'is_llm_role']


,count,mean,std,min,25%,50%,75%,max
years_of_experience,1500.0,6.216000,2.675216,1.0,4.000,6.0,8.0,15.0
annual_salary_usd,1500.0,194892.000000,66506.822013,90000.0,144750.000,180000.0,236250.0,384000.0
salary_min_usd,1500.0,135448.666667,24448.950878,90000.0,120000.000,140000.0,155000.0,180000.0
salary_max_usd,1500.0,257537.333333,39852.822207,180000.0,218000.000,270000.0,290000.0,320000.0
ai_salary_premium_pct,1500.0,10.858200,4.029742,3.0,8.200,10.5,14.2,18.0
demand_score,1500.0,87.523333,8.026315,68.0,82.000,89.0,95.0,98.0
demand_growth_yoy_pct,1500.0,31.116333,22.046343,5.0,15.375,23.4,42.7,87.8
benefits_score_10,1500.0,7.897333,1.102846,6.0,6.900,7.9,8.9,9.8
posting_year,1500.0,2025.584000,0.493058,2025.0,2025.000,2026.0,2026.0,2026.0
posting_month,1500.0,3.968000,3.270388,1.0,2.000,3.0,5.0,12.0


Step 9: Check Categorical Consistency
Examine unique categorical values and their frequencies to identify inconsistent labels or unexpected categories.

In [38]:

categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print("Unique Values:", df[col].nunique())
    print(df[col].value_counts(dropna=False).head(15))


--- job_id ---
Unique Values: 1500
job_id
AIJOB1500    1
AIJOB0001    1
AIJOB0002    1
AIJOB0003    1
AIJOB0004    1
AIJOB0005    1
AIJOB0006    1
AIJOB0007    1
AIJOB0008    1
AIJOB0009    1
AIJOB0010    1
AIJOB0011    1
AIJOB1484    1
AIJOB1483    1
AIJOB1482    1
Name: count, dtype: int64

--- job_title ---
Unique Values: 25
job_title
LLM Engineer              75
Robotics Engineer (AI)    74
Prompt Engineer           71
Generative AI Engineer    71
AI Product Manager        70
Multimodal AI Engineer    67
Senior Data Scientist     66
AI Compliance Manager     66
Senior ML Engineer        64
AI Engineer               64
AI Business Analyst       62
Data Scientist            61
Deep Learning Engineer    58
AI Agent Developer        57
AI Ethics Officer         56
Name: count, dtype: int64

--- job_category ---
Unique Values: 12
job_category
AI Engineering      736
Data Science        127
Governance          122
Robotics             74
Product              70
Business             62
I

Step 10: Initial Data Quality Summary
Generate a final overview of the dataset after the initial cleaning steps to identify remaining data-quality issues before feature engineering and visualization.

In [39]:

print("Dataset Shape:", df.shape)

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

print("\nNumeric Summary:")
display(df.describe(include='all').T)

Dataset Shape: (1500, 25)

Duplicate Rows: 0

Missing Values:
job_id                   0
job_title                0
job_category             0
experience_level         0
years_of_experience      0
education_required       0
annual_salary_usd        0
salary_min_usd           0
salary_max_usd           0
city                     0
country                  0
remote_work              0
company_size             0
industry                 0
required_skills          0
ai_salary_premium_pct    0
demand_score             0
demand_growth_yoy_pct    0
benefits_score_10        0
posting_year             0
posting_month            0
is_senior                0
is_remote_friendly       0
is_llm_role              0
salary_tier              0
dtype: int64

Data Types:
job_id                    object
job_title                 object
job_category              object
experience_level          object
years_of_experience        int64
education_required        object
annual_salary_usd        float64
salary

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
job_id,1500,1500,AIJOB1500,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
job_title,1500,25,LLM Engineer,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
job_category,1500,12,AI Engineering,736,NaN,NaN,NaN,NaN,NaN,NaN,NaN
experience_level,1500,4,Entry (0-2 yrs),385,NaN,NaN,NaN,NaN,NaN,NaN,NaN
years_of_experience,1500.0,NaN,NaN,NaN,6.216,2.675216,1.0,4.0,6.0,8.0,15.0
education_required,1500,5,Master's,316,NaN,NaN,NaN,NaN,NaN,NaN,NaN
annual_salary_usd,1500.0,NaN,NaN,NaN,194892.0,66506.822013,90000.0,144750.0,180000.0,236250.0,384000.0
salary_min_usd,1500.0,NaN,NaN,NaN,135448.666667,24448.950878,90000.0,120000.0,140000.0,155000.0,180000.0
salary_max_usd,1500.0,NaN,NaN,NaN,257537.333333,39852.822207,180000.0,218000.0,270000.0,290000.0,320000.0
city,1500,20,San Francisco,92,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:

invalid_salary_ranges = df[
    df['salary_min_usd'] > df['salary_max_usd']
]

print("Number of Invalid Salary Ranges:", len(invalid_salary_ranges))

invalid_salary_ranges[
    ['job_title', 'salary_min_usd', 'salary_max_usd']
].head(10)

Number of Invalid Salary Ranges: 0


,job_title,salary_min_usd,salary_max_usd


Step 12: Validate Numeric Ranges
Check numerical columns against their expected ranges to identify invalid or unrealistic values before feature engineering.

In [41]:

print("Invalid Years of Experience:",
      (df['years_of_experience'] < 0).sum())

print("Invalid Minimum Salary:",
      (df['salary_min_usd'] <= 0).sum())

print("Invalid Maximum Salary:",
      (df['salary_max_usd'] <= 0).sum())

print("Invalid AI Salary Premium:",
      (df['ai_salary_premium_pct'] < 0).sum())

print("Invalid Demand Score:",
      ((df['demand_score'] < 0) | (df['demand_score'] > 100)).sum())

print("Invalid Demand Growth:",
      (df['demand_growth_yoy_pct'] < 0).sum())

print("Invalid Benefits Score:",
      ((df['benefits_score_10'] < 0) | (df['benefits_score_10'] > 10)).sum())

print("Invalid Posting Month:",
      ((df['posting_month'] < 1) | (df['posting_month'] > 12)).sum())

Invalid Years of Experience: 0
Invalid Minimum Salary: 0
Invalid Maximum Salary: 0
Invalid AI Salary Premium: 0
Invalid Demand Score: 0
Invalid Demand Growth: 0
Invalid Benefits Score: 0
Invalid Posting Month: 0


Step 13: Salary Feature Engineering
Create derived salary features to support salary distribution, comparison, and range analysis during visualization.

In [42]:

df['salary_range_width_usd'] = (
    df['salary_max_usd'] - df['salary_min_usd']
)

df['salary_range_midpoint_usd'] = (
    df['salary_min_usd'] + df['salary_max_usd']
) / 2

df['salary_range_valid'] = (
    df['salary_min_usd'] <= df['salary_max_usd']
)

print("Salary Range Width - Summary:")
print(df['salary_range_width_usd'].describe())

print("\nSalary Range Midpoint - Summary:")
print(df['salary_range_midpoint_usd'].describe())

print("\nInvalid Salary Ranges:",
      (~df['salary_range_valid']).sum())

Salary Range Width - Summary:
count      1500.000000
mean     122088.666667
std       27184.115257
min       57000.000000
25%      110000.000000
50%      130000.000000
75%      140000.000000
max      165000.000000
Name: salary_range_width_usd, dtype: float64

Salary Range Midpoint - Summary:
count      1500.000000
mean     196493.000000
std       30137.274062
min      137500.000000
25%      175000.000000
50%      200000.000000
75%      222500.000000
max      250000.000000
Name: salary_range_midpoint_usd, dtype: float64

Invalid Salary Ranges: 0


Step 14: Validate Annual Salary Consistency
Verify whether the annual salary falls within the reported minimum and maximum salary range.

In [43]:

df['annual_salary_within_posted_range'] = (
    (df['annual_salary_usd'] >= df['salary_min_usd']) &
    (df['annual_salary_usd'] <= df['salary_max_usd'])
)

print("Annual Salary Within Posted Range:")
print(df['annual_salary_within_posted_range'].value_counts())

print("\nAnnual Salary Outside Posted Range:",
      (~df['annual_salary_within_posted_range']).sum())

Annual Salary Within Posted Range:
annual_salary_within_posted_range
True     1212
False     288
Name: count, dtype: int64

Annual Salary Outside Posted Range: 288


Step 15: Skill Count Feature
Calculate the number of required skills for each job posting based on the pipe-separated skills list.

In [44]:

df['skill_count'] = (
    df['required_skills']
    .astype(str)
    .str.split('|')
    .apply(len)
)

print("Skill Count Summary:")
print(df['skill_count'].describe())

print("\nMinimum Skill Count:", df['skill_count'].min())
print("Maximum Skill Count:", df['skill_count'].max())

Skill Count Summary:
count    1500.000000
mean        6.365333
std         1.355393
min         4.000000
25%         5.000000
50%         6.000000
75%         7.000000
max        10.000000
Name: skill_count, dtype: float64

Minimum Skill Count: 4
Maximum Skill Count: 10


Step 16: Create Posting Date
Combine posting year and month into a standardized date feature for time-based analysis and visualization.

In [45]:

df['posting_date'] = pd.to_datetime(
    df['posting_year'].astype(str) + '-' +
    df['posting_month'].astype(str) + '-01',
    errors='coerce'
)

print("Invalid Posting Dates:",
      df['posting_date'].isna().sum())

print("\nPosting Date Range:")
print(df['posting_date'].min(), "to", df['posting_date'].max())

Invalid Posting Dates: 0

Posting Date Range:
2025-01-01 00:00:00 to 2026-03-01 00:00:00


Step 17: Validate Posting Date
Verify that the generated posting date correctly represents the original posting year and month values.

In [46]:

date_valid = (
    (df['posting_date'].dt.year == df['posting_year']) &
    (df['posting_date'].dt.month == df['posting_month'])
)

print("Valid Posting Dates:", date_valid.sum())
print("Invalid Posting Dates:", (~date_valid).sum())

Valid Posting Dates: 1500
Invalid Posting Dates: 0


Step 18: Outlier Detection Using IQR
Identify potential outliers in key numerical variables using the Interquartile Range (IQR) method without removing valid observations.

In [47]:

def iqr_outlier(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    return (series < lower_bound) | (series > upper_bound)


df['experience_iqr_outlier'] = iqr_outlier(
    df['years_of_experience']
)

df['salary_iqr_outlier'] = iqr_outlier(
    df['annual_salary_usd']
)

df['demand_growth_iqr_outlier'] = iqr_outlier(
    df['demand_growth_yoy_pct']
)

print("Experience IQR Outliers:",
      df['experience_iqr_outlier'].sum())

print("Salary IQR Outliers:",
      df['salary_iqr_outlier'].sum())

print("Demand Growth IQR Outliers:",
      df['demand_growth_iqr_outlier'].sum())

Experience IQR Outliers: 5
Salary IQR Outliers: 8
Demand Growth IQR Outliers: 29


Step 19: Outlier Detection Using Z-Score
Identify extreme numerical observations using Z-score analysis and flag them for further exploration without removing valid observations.

In [48]:

from scipy.stats import zscore
import numpy as np

df['experience_zscore_outlier'] = (
    np.abs(zscore(df['years_of_experience'])) > 3
)

df['salary_zscore_outlier'] = (
    np.abs(zscore(df['annual_salary_usd'])) > 3
)

df['demand_growth_zscore_outlier'] = (
    np.abs(zscore(df['demand_growth_yoy_pct'])) > 3
)

print("Experience Z-Score Outliers:",
      df['experience_zscore_outlier'].sum())

print("Salary Z-Score Outliers:",
      df['salary_zscore_outlier'].sum())

print("Demand Growth Z-Score Outliers:",
      df['demand_growth_zscore_outlier'].sum())

Experience Z-Score Outliers: 5
Salary Z-Score Outliers: 0
Demand Growth Z-Score Outliers: 0


Step 20: Categorical Value Consistency
Review categorical columns to ensure that their values are consistent and contain no unexpected categories.

In [49]:

categorical_cols = [
    'job_category',
    'experience_level',
    'education_required',
    'remote_work',
    'company_size',
    'industry',
    'salary_tier'
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print("Unique Values:", df[col].nunique())
    print(df[col].value_counts())


--- job_category ---
Unique Values: 12
job_category
AI Engineering      736
Data Science        127
Governance          122
Robotics             74
Product              70
Business             62
Infrastructure       55
Architecture         52
ML Operations        51
Data Engineering     51
Security             50
Research             50
Name: count, dtype: int64

--- experience_level ---
Unique Values: 4
experience_level
Entry (0-2 yrs)     385
Lead (10+ yrs)      381
Mid (3-5 yrs)       370
Senior (6-9 yrs)    364
Name: count, dtype: int64

--- education_required ---
Unique Values: 5
education_required
Master's                316
Bachelor's              311
Bootcamp/Self-taught    297
Associate's             296
PhD                     280
Name: count, dtype: int64

--- remote_work ---
Unique Values: 3
remote_work
Hybrid          686
Fully Remote    445
On-site         369
Name: count, dtype: int64

--- company_size ---
Unique Values: 5
company_size
Mid-size (501-5000)    312
SME (5

Step 21: Validate Binary Features
Verify that binary indicator columns contain only valid 0 and 1 values.

In [50]:

binary_cols = [
    'is_senior',
    'is_remote_friendly',
    'is_llm_role'
]

for col in binary_cols:
    print(f"\n--- {col} ---")
    print("Unique Values:", sorted(df[col].unique()))
    print(df[col].value_counts())


--- is_senior ---
Unique Values: [np.int64(0), np.int64(1)]
is_senior
0    755
1    745
Name: count, dtype: int64

--- is_remote_friendly ---
Unique Values: [np.int64(0), np.int64(1)]
is_remote_friendly
1    1131
0     369
Name: count, dtype: int64

--- is_llm_role ---
Unique Values: [np.int64(0), np.int64(1)]
is_llm_role
0    1173
1     327
Name: count, dtype: int64


Step 22: Validate Numerical Data Types
Verify that numerical features use appropriate numeric data types for analysis and visualization.

In [51]:

numeric_cols = [
    'years_of_experience',
    'annual_salary_usd',
    'salary_min_usd',
    'salary_max_usd',
    'ai_salary_premium_pct',
    'demand_score',
    'demand_growth_yoy_pct',
    'benefits_score_10',
    'posting_year',
    'posting_month',
    'is_senior',
    'is_remote_friendly',
    'is_llm_role',
    'salary_range_width_usd',
    'salary_range_midpoint_usd',
    'skill_count'
]

print(df[numeric_cols].dtypes)

years_of_experience            int64
annual_salary_usd            float64
salary_min_usd                 int64
salary_max_usd                 int64
ai_salary_premium_pct        float64
demand_score                   int64
demand_growth_yoy_pct        float64
benefits_score_10            float64
posting_year                   int64
posting_month                  int64
is_senior                      int64
is_remote_friendly             int64
is_llm_role                    int64
salary_range_width_usd         int64
salary_range_midpoint_usd    float64
skill_count                    int64
dtype: object


Step 23: Final Missing Values Check
Verify that the cleaned dataset contains no missing values before visualization and export.

In [52]:

missing_values = df.isnull().sum()

print("Columns with Missing Values:")
print(missing_values[missing_values > 0])

print("\nTotal Missing Values:", df.isnull().sum().sum())

Columns with Missing Values:
Series([], dtype: int64)

Total Missing Values: 0


Step 24: Final Duplicate Check
Verify that the processed dataset contains no duplicate records before visualization and export.

In [53]:

duplicate_count = df.duplicated().sum()

print("Number of Duplicate Rows:", duplicate_count)

Number of Duplicate Rows: 0


Step 25: Final Dataset Shape
Verify the final number of rows and columns after completing data cleaning and feature engineering.

In [54]:


print("Final Dataset Shape:", df.shape)
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

Final Dataset Shape: (1500, 37)
Number of Rows: 1500
Number of Columns: 37


Step 26: Export Final Processed Dataset
Save the fully processed dataset as a CSV file for visualization and further analysis.

In [55]:


output_file = 'ai_jobs_global_processed.csv'

df.to_csv(output_file, index=False)

print("Dataset exported successfully.")
print("File name:", output_file)
print("Final Shape:", df.shape)

Dataset exported successfully.
File name: ai_jobs_global_processed.csv
Final Shape: (1500, 37)


In [56]:
from google.colab import files

files.download('ai_jobs_global_processed.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>